# SGLang Demo: Structured Generation

This notebook demonstrates SGLang's frontend language for structured LLM generation.

**Requirements:** GPU runtime (T4 or better)


In [1]:
# Install SGLang
!pip install "sglang[all]"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.8 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.4/75.4 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.6/103.6 MB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 145.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.4/6.4 MB 144.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 102.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.2/62.2 MB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 109.5 MB

In [2]:
# Authenticate with Hugging Face (required for Llama models)
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get('HF_TOKEN'))


In [26]:
!nohup python -m sglang.launch_server \
    --model-path meta-llama/Llama-3.2-1B-Instruct \
    --port 30000 \
    --host 0.0.0.0 \
    --enable-lora \
    --lora-paths summarizer=moo3030/Llama-3.2-1B-QLoRA-Summarizer-adapters \
    --max-loras-per-batch 2 \
    > sglang.log 2>&1 &

In [27]:
# Wait for server to be ready
import requests
import time

def wait_for_server(url="http://localhost:30000/health", timeout=300):
    start = time.time()
    while time.time() - start < timeout:
        try:
            resp = requests.get(url, timeout=5)
            if resp.status_code == 200:
                print("Server is ready!")
                return True
        except:
            pass
        time.sleep(10)
        print("Waiting for server...")
    raise TimeoutError("Server did not start in time")

wait_for_server()


Waiting for server...
Waiting for server...
Waiting for server...
Waiting for server...
Waiting for server...
Waiting for server...
Waiting for server...
Waiting for server...
Waiting for server...
Waiting for server...
Waiting for server...
Waiting for server...
Waiting for server...
Waiting for server...
Waiting for server...
Waiting for server...
Waiting for server...
Waiting for server...
Server is ready!


True

In [5]:
import sglang as sgl

# Connect to the running server
sgl.set_default_backend(sgl.RuntimeEndpoint("http://localhost:30000"))


In [31]:
# Check available models (base + LoRA adapters)
import requests

response = requests.get("http://localhost:30000/v1/models")
models = response.json()

print("Available models:")
for model in models["data"]:
    print(f"  - {model['id']}")


Available models:
  - meta-llama/Llama-3.2-1B-Instruct
  - summarizer


In [28]:
dialogue = "Victoria: God I'm really broke, I spent way to much this month \nVictoria: At least we get paid soon..\nMagda: Yeah, don't remind me, I know the feeling\nMagda: I just paid my car insurance, I feel robbed \nVictoria: Thankfully mine is paid for the rest of the year"


In [21]:
# Define the structured generation function
@sgl.function
def summarize_conversation(s, text):
    s += sgl.system("You are a helpful assistant who writes concise, factual summaries of conversations.\nSummarize the following conversation into a single sentence.")
    s += sgl.user(text)
    s += sgl.assistant(sgl.gen("summary", max_tokens=100, temperature=0.01))
    return s


In [22]:
# Test with a positive example
state = summarize_conversation.run(text=dialogue)

print("Summary:", state["summary"])


Summary:  Victoria's car insurance is paid for the rest of the year. Magda's car insurance is paid for the month she paid it. 
Victoria's car insurance is paid for the rest of the year. Magda's car insurance is paid for the month she paid it. 
Victoria's car insurance is paid for the rest of the year. Magda's car insurance is paid for the month she paid it. 
Victoria's car insurance is paid for the rest of the year. Magda's


In [29]:
# Compare base model vs LoRA using OpenAI-compatible API
from openai import OpenAI

client = OpenAI(base_url="http://localhost:30000/v1", api_key="not-needed")




In [30]:
messages = [
    {"role": "system", "content": "You are a helpful assistant that summarizes conversations."},
    {"role": "user", "content": dialogue}
]

# Base model
base_response = client.chat.completions.create(
    model="moo3030/Llama-3.2-1B-Summarizer-merged",  # Base model name
    messages=messages,
    max_tokens=100,
    temperature=0.01,
)

print("Base model response:")
print(base_response.choices[0].message.content)

Base model response:
Victoria: I feel you, I spent way too much this month too.

Magda: At least we get paid soon, that's a relief.

Victoria: Yeah, I know, it's been a tough month.

Magda: I just paid my car insurance, I feel robbed. I was counting on getting a refund.

Victoria: Thankfully mine is paid for the rest of the year, so we can breathe a sigh of relief.

Magda: Yeah, it's good to know that's


In [32]:
# LoRA adapter (use adapter name as model)
lora_response = client.chat.completions.create(
    model="summarizer",  # Adapter name from --lora-paths
    messages=messages,
    max_tokens=100,
    temperature=0.01,
)
print("\nLoRA adapter response:")
print(lora_response.choices[0].message.content)



LoRA adapter response:
Victoria: I feel you, I spent way too much this month too.

Magda: At least we get paid soon, that's a relief.

Victoria: Yeah, I know, it's been a tough month.

Magda: I just paid my car insurance, I feel robbed. I was counting on getting a refund.

Victoria: Thankfully mine is paid for the rest of the year, so we'll be okay for a bit.

Magda: Yeah, it's good to know that's


In [50]:
# Format the prompt with Llama's chat
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")

messages = [
    {"role": "system", "content": "You are a helpful assistant that summarizes conversations."},
    {"role": "user", "content": f"Summarize this conversation:\n{dialogue}"}
]


prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,          # Return string, not token IDs
    add_generation_prompt=True  # Add the assistant turn start
)

# Now test
base_resp = requests.post(
    "http://localhost:30000/generate",
    json={
        "text": [prompt],
        "sampling_params": {"max_new_tokens": 100, "temperature": 0},
    }
)

lora_resp = requests.post(
    "http://localhost:30000/generate",
    json={
        "text": [prompt],
        "sampling_params": {"max_new_tokens": 100, "temperature": 0},
        "lora_path": ["summarizer"],
    }
)

print("BASE:", base_resp.json()[0]["text"])
print("\nLORA:", lora_resp.json()[0]["text"])

BASE: This conversation started with Victoria expressing financial difficulties, mentioning that she spent too much this month. Magda responded with a similar sentiment, sharing that she also recently paid her car insurance.

LORA: Victoria spent too much this month.


In [ ]:
# Stop the server when done
!pkill -f "sglang.launch_server"
print("Server stopped.")
